# CA-LMoETransUNet — Canada Wildfire Training

**Steps:**
1. Mount Google Drive
2. Upload `patches_ca.npz` to your Drive at `MyDrive/wildfire/patches_ca.npz`
3. Run all cells — ~15 min on T4 GPU for 20 epochs


In [ ]:
# ── Cell 1: Mount Google Drive ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
NPZ_PATH = '/content/drive/MyDrive/wildfire/patches_ca.npz'
OUT_DIR  = '/content/drive/MyDrive/wildfire/checkpoints_ca'
os.makedirs(OUT_DIR, exist_ok=True)

# Verify file exists
if os.path.exists(NPZ_PATH):
    size_gb = os.path.getsize(NPZ_PATH) / 1e9
    print(f'Found patches_ca.npz  ({size_gb:.1f} GB)')
else:
    print('ERROR: patches_ca.npz not found!')
    print(f'Please upload it to Google Drive at: {NPZ_PATH}')

In [ ]:
# ── Cell 2: Check GPU ──────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Cell 3: Model Definition ───────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.drop  = nn.Dropout(dropout)
        self.short = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch else nn.Sequential()

    def forward(self, x):
        s = self.short(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.drop(x)
        x = self.bn2(self.conv2(x))
        return self.relu(x + s)

class MoEAttentionBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_dim, num_experts=3, dropout=0.4):
        super().__init__()
        self.num_heads   = num_heads
        self.num_experts = num_experts
        self.qkv_proj    = nn.ModuleList([nn.Linear(dim, dim * 3) for _ in range(num_experts)])
        self.out_proj    = nn.ModuleList([nn.Linear(dim, dim)     for _ in range(num_experts)])
        self.gate        = nn.Linear(dim, num_experts)
        self.drop        = nn.Dropout(dropout)
        self.norm1       = nn.LayerNorm(dim)
        self.norm2       = nn.LayerNorm(dim)
        self.mlp         = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.ReLU(inplace=True), nn.Linear(mlp_dim, dim)
        )

    def forward(self, x):
        B, L, D = x.shape
        xn = self.norm1(x)
        gate_probs = F.softmax(self.gate(xn), dim=-1)
        expert_outs = []
        for i in range(self.num_experts):
            q, k, v = self.qkv_proj[i](xn).chunk(3, dim=-1)
            q = F.relu(q.contiguous().view(B, L, self.num_heads, -1).transpose(1, 2))
            k = F.relu(k.contiguous().view(B, L, self.num_heads, -1).transpose(1, 2))
            v =        v.contiguous().view(B, L, self.num_heads, -1).transpose(1, 2)
            kv   = torch.einsum('bnld,bnle->bnle', k, v)
            z    = 1 / (torch.einsum('bnld,bnld->bnl', q, k) + 1e-6).unsqueeze(-1)
            attn = torch.einsum('bnld,bnle->bnle', q, kv) * z
            attn = attn.transpose(1, 2).reshape(B, L, D)
            expert_outs.append(self.drop(self.out_proj[i](attn)))
        combined = sum(gate_probs[..., i:i+1] * expert_outs[i] for i in range(self.num_experts))
        x = x + self.drop(combined)
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

class CALMoETransUNet(nn.Module):
    def __init__(self, in_channels=12, out_channels=2,
                 embed_dim=128, num_heads=4, mlp_dim=256,
                 transformer_depth=1, num_experts=3):
        super().__init__()
        self.enc1 = ResidualBlock(in_channels, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.enc3 = ResidualBlock(128, 256)
        self.enc4 = ResidualBlock(256, embed_dim)
        self.flat = nn.Flatten(2)
        self.transformers = nn.ModuleList([
            MoEAttentionBlock(embed_dim, num_heads, mlp_dim, num_experts)
            for _ in range(transformer_depth)
        ])
        self.up4  = nn.ConvTranspose2d(embed_dim, 256, 2, stride=2)
        self.dec4 = ResidualBlock(256, 256)
        self.up3  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = ResidualBlock(128, 128)
        self.up2  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = ResidualBlock(64, 64)
        self.head = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        e4 = self.enc4(F.max_pool2d(e3, 2))
        b, c, h, w = e4.shape
        t = self.flat(e4).permute(0, 2, 1)
        for blk in self.transformers:
            t = blk(t)
        t  = t.permute(0, 2, 1).view(b, c, h, w)
        d4 = self.dec4(self.up4(t)  + e3)
        d3 = self.dec3(self.up3(d4) + e2)
        d2 = self.dec2(self.up2(d3) + e1)
        return self.head(d2)
print('Model defined OK')

In [ ]:
# ── Cell 4: Extract NPZ and Memory Map to fix RAM limits ────────────────────
import numpy as np
import random
import os
from torch.utils.data import Dataset, DataLoader, random_split

# 1. Unzip .npz locally to /content so we can memory map it (avoids RAM crash!)
import zipfile
print('Extracting patches_ca.npz to local disk... (this takes ~1-2 mins)')
EXTRACT_DIR = '/content/extracted_patches'
os.makedirs(EXTRACT_DIR, exist_ok=True)

if not os.path.exists(os.path.join(EXTRACT_DIR, 'images.npy')):
    with zipfile.ZipFile(NPZ_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
print('Extraction complete!')

# 2. Use mmap_mode='r' so we don't load 16GB into RAM
print('Memory mapping arrays...')
images_mmap = np.load(os.path.join(EXTRACT_DIR, 'images.npy'), mmap_mode='r')
masks_mmap  = np.load(os.path.join(EXTRACT_DIR, 'masks.npy'), mmap_mode='r')
print(f'Mapped: images={images_mmap.shape}  masks={masks_mmap.shape}')

class MmapDataset(Dataset):
    def __init__(self, images, masks, augment=True):
        self.images  = images
        self.masks   = masks
        self.augment = augment
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        # Load only the requested patch from disk into RAM
        img  = torch.from_numpy(self.images[idx].copy())
        mask = torch.from_numpy(self.masks[idx].astype(np.int64))
        # Clean NaN pixels (water-masked areas)
        img = torch.nan_to_num(img, nan=0.0, posinf=1.0, neginf=0.0).clamp(0.0, 1.0)
        if self.augment:
            if random.random() > 0.5:
                img  = torch.flip(img,  dims=[-1])
                mask = torch.flip(mask, dims=[-1])
            if random.random() > 0.5:
                img  = torch.flip(img,  dims=[-2])
                mask = torch.flip(mask, dims=[-2])
        return img, mask

# Train / val split
N        = len(images_mmap)
val_size = max(1, int(N * 0.2))
tr_size  = N - val_size
full_ds  = MmapDataset(images_mmap, masks_mmap)
train_ds, val_ds = random_split(full_ds, [tr_size, val_size],
                                generator=torch.Generator().manual_seed(42))

BATCH = 16
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {tr_size}  Val: {val_size}  | Batches/epoch: {len(train_loader)}')

In [ ]:
# ── Cell 5: Training Setup ─────────────────────────────────────────
import os, time

EPOCHS     = 20
LR         = 1e-4
EMBED_DIM  = 128
NUM_EXPERT = 3
DEPTH      = 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

model = CALMoETransUNet(
    in_channels=12, out_channels=2,
    embed_dim=EMBED_DIM, num_heads=4,
    mlp_dim=EMBED_DIM*2,
    transformer_depth=DEPTH,
    num_experts=NUM_EXPERT
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {total_params:,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

def dice_loss(pred, target, smooth=1.0):
    prob   = torch.softmax(pred, dim=1)[:, 1]
    target = target.float()
    inter  = (prob * target).sum()
    return 1 - (2*inter + smooth) / (prob.sum() + target.sum() + smooth)

ce_loss = nn.CrossEntropyLoss()
print('Setup complete!')

In [ ]:
# ── Cell 6: Train! ─────────────────────────────────────────────
best_dice = 0.0
history   = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss, tr_dice = 0.0, 0.0
    t0 = time.time()
    for imgs, masks_b in train_loader:
        imgs, masks_b = imgs.to(device), masks_b.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = ce_loss(logits, masks_b) + dice_loss(logits, masks_b)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()
        with torch.no_grad():
            pred  = logits.argmax(1)
            inter = ((pred==1) & (masks_b==1)).float().sum()
            denom = (pred==1).float().sum() + (masks_b==1).float().sum() + 1e-6
            tr_dice += (2*inter/denom).item()

    model.eval()
    vl_loss, vl_dice, vl_iou = 0.0, 0.0, 0.0
    with torch.no_grad():
        for imgs, masks_b in val_loader:
            imgs, masks_b = imgs.to(device), masks_b.to(device)
            logits = model(imgs)
            loss   = ce_loss(logits, masks_b) + dice_loss(logits, masks_b)
            vl_loss += loss.item()
            pred  = logits.argmax(1)
            inter = ((pred==1) & (masks_b==1)).float().sum()
            union = ((pred==1) | (masks_b==1)).float().sum()
            denom = (pred==1).float().sum() + (masks_b==1).float().sum() + 1e-6
            vl_dice += (2*inter/denom).item()
            vl_iou  += (inter/(union+1e-6)).item()

    scheduler.step()
    n_tr, n_vl = len(train_loader), len(val_loader)
    elapsed    = time.time() - t0

    row = dict(epoch=epoch, tr_loss=tr_loss/n_tr, tr_dice=tr_dice/n_tr,
               vl_loss=vl_loss/n_vl, vl_dice=vl_dice/n_vl, vl_iou=vl_iou/n_vl)
    history.append(row)

    print(f"Epoch {epoch:03d}/{EPOCHS}  "
          f"train loss={row['tr_loss']:.4f} dice={row['tr_dice']:.4f}  |  "
          f"val loss={row['vl_loss']:.4f} dice={row['vl_dice']:.4f} IoU={row['vl_iou']:.4f}  "
          f"[{elapsed:.0f}s]")

    if row['vl_dice'] > best_dice:
        best_dice = row['vl_dice']
        ckpt_path = os.path.join(OUT_DIR, 'best_model.pth')
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_dice': vl_dice/n_vl, 'val_iou': vl_iou/n_vl}, ckpt_path)
        print(f'  ✓ Saved best model (dice={best_dice:.4f}) -> {ckpt_path}')

print(f'\nTraining complete! Best val Dice: {best_dice:.4f}')

In [ ]:
# ── Cell 7: Plot Training Curves ───────────────────────────────────
import matplotlib.pyplot as plt

epochs_x  = [h['epoch']   for h in history]
tr_dice_y = [h['tr_dice'] for h in history]
vl_dice_y = [h['vl_dice'] for h in history]
vl_iou_y  = [h['vl_iou']  for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_x, tr_dice_y, 'b-o', label='Train Dice')
ax1.plot(epochs_x, vl_dice_y, 'r-o', label='Val Dice')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Dice Score')
ax1.set_title('Dice Score over Epochs')
ax1.legend(); ax1.grid(True)

ax2.plot(epochs_x, vl_iou_y, 'g-o', label='Val IoU')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('IoU')
ax2.set_title('Validation IoU over Epochs')
ax2.legend(); ax2.grid(True)

plt.suptitle('CA-LMoETransUNet — Canada Wildfire', fontsize=13)
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved: {plot_path}')

In [ ]:
# ── Cell 8: Download best model to local machine ────────────────────
from google.colab import files
files.download(os.path.join(OUT_DIR, 'best_model.pth'))